# Filter SigLIP Embedding

This notebook filters SigLIP embeddings to match the paths from CLIP keyframe embeddings.

## Input:
- `all_keyframes_combined.pkl`: CLIP embeddings with keyframe paths
- `LIST_SIGLIP_EMBEDDING`: List of SigLIP embedding pkl files

## Output:
- Filtered CLIP and SigLIP embeddings with same paths and length

## Data Structure:
```python
{
    "paths": [...],
    "embeddings": [...],
    "length": int
}
```

In [1]:
import pickle
import numpy as np
from pathlib import Path
from tqdm import tqdm
import json
from collections import defaultdict

In [2]:
class SigLIPEmbeddingFilter:
    """
    Class for filtering SigLIP embeddings based on CLIP keyframe paths.
    """
    
    def __init__(self):
        self.clip_data = None
        self.siglip_data = None
    
    def load_clip_keyframes(self, clip_file_path):
        """
        Load CLIP keyframe embeddings from pkl file.
        
        Args:
            clip_file_path (str): Path to all_keyframes_combined.pkl
        """
        print(f"📂 Loading CLIP keyframes from {clip_file_path}...")
        
        with open(clip_file_path, 'rb') as f:
            self.clip_data = pickle.load(f)
        
        print(f"✅ CLIP data loaded:")
        print(f"   📊 Paths: {len(self.clip_data['paths']):,}")
        print(f"   📊 Embeddings: {len(self.clip_data['embeddings']):,}")
        print(f"   📊 Length: {self.clip_data['length']:,}")
        
        # Validate data consistency
        if len(self.clip_data['paths']) != len(self.clip_data['embeddings']):
            raise ValueError(f"CLIP data inconsistent: {len(self.clip_data['paths'])} paths != {len(self.clip_data['embeddings'])} embeddings")
    
    def load_siglip_embeddings(self, siglip_file_list):
        """
        Load and combine multiple SigLIP embedding files.
        
        Args:
            siglip_file_list (list): List of paths to SigLIP embedding pkl files
        """
        print(f"📂 Loading {len(siglip_file_list)} SigLIP embedding files...")
        
        combined_siglip = {
            "paths": [],
            "embeddings": [],
            "length": 0
        }
        
        for i, siglip_file in enumerate(siglip_file_list, 1):
            print(f"   📄 Loading file {i}/{len(siglip_file_list)}: {siglip_file}")
            
            try:
                with open(siglip_file, 'rb') as f:
                    siglip_data = pickle.load(f)
                
                # Validate structure
                if not all(key in siglip_data for key in ['paths', 'embeddings', 'length']):
                    raise ValueError(f"Missing required keys in {siglip_file}")
                
                if len(siglip_data['paths']) != len(siglip_data['embeddings']):
                    raise ValueError(f"Inconsistent data in {siglip_file}: {len(siglip_data['paths'])} paths != {len(siglip_data['embeddings'])} embeddings")
                
                # Combine data
                combined_siglip["paths"].extend(siglip_data["paths"])
                combined_siglip["embeddings"].extend(siglip_data["embeddings"])
                combined_siglip["length"] += len(siglip_data["paths"])
                
                print(f"      ✅ Added {len(siglip_data['paths']):,} embeddings")
                
            except Exception as e:
                print(f"      ❌ Error loading {siglip_file}: {e}")
                raise
        
        self.siglip_data = combined_siglip
        
        print(f"\n✅ Combined SigLIP data:")
        print(f"   📊 Total paths: {len(self.siglip_data['paths']):,}")
        print(f"   📊 Total embeddings: {len(self.siglip_data['embeddings']):,}")
        print(f"   📊 Total length: {self.siglip_data['length']:,}")
    
    def filter_siglip_by_clip_paths(self):
        """
        Filter SigLIP embeddings to match CLIP keyframe paths.
        
        Returns:
            dict: filtered_siglip_data with same paths as CLIP
        """
        if self.clip_data is None:
            raise ValueError("CLIP data not loaded. Call load_clip_keyframes() first.")
        
        if self.siglip_data is None:
            raise ValueError("SigLIP data not loaded. Call load_siglip_embeddings() first.")
        
        print(f"🔄 Filtering SigLIP embeddings to match CLIP paths...")
        
        # Create path-to-embedding mapping for SigLIP
        siglip_path_to_embedding = {}
        for path, embedding in zip(self.siglip_data["paths"], self.siglip_data["embeddings"]):
            siglip_path_to_embedding[path] = embedding
        
        print(f"   📊 SigLIP path lookup created: {len(siglip_path_to_embedding):,} entries")
        
        # Create filtered SigLIP data based on CLIP paths
        filtered_siglip_data = {
            "paths": [],
            "embeddings": [],
            "length": 0
        }
        
        missing_paths = []
        
        print(f"   🔍 Filtering SigLIP embeddings...")
        for clip_path in tqdm(self.clip_data["paths"], desc="Filtering"):
            if clip_path in siglip_path_to_embedding:
                # Add corresponding SigLIP embedding
                filtered_siglip_data["paths"].append(clip_path)
                filtered_siglip_data["embeddings"].append(siglip_path_to_embedding[clip_path])
                filtered_siglip_data["length"] += 1
            else:
                # Path not found in SigLIP
                missing_paths.append(clip_path)
        
        # Print statistics
        print(f"\n📊 Filtering results:")
        print(f"   ✅ Matched paths: {filtered_siglip_data['length']:,}")
        print(f"   ❌ Missing paths: {len(missing_paths):,}")
        print(f"   📈 Match rate: {(filtered_siglip_data['length'] / len(self.clip_data['paths']) * 100):.1f}%")
        
        if missing_paths:
            print(f"\n⚠️  Sample missing paths:")
            for path in missing_paths[:5]:
                print(f"      📄 {path}")
            if len(missing_paths) > 5:
                print(f"      ... and {len(missing_paths) - 5} more")
        
        return filtered_siglip_data
    
    def save_filtered_siglip(self, filtered_siglip_data, siglip_output_path):
        """
        Save filtered SigLIP data to file.
        
        Args:
            filtered_siglip_data (dict): Filtered SigLIP data
            siglip_output_path (str): Output path for filtered SigLIP data
        """
        print(f"💾 Saving filtered SigLIP data...")
        
        # Save SigLIP data
        with open(siglip_output_path, 'wb') as f:
            pickle.dump(filtered_siglip_data, f)
        print(f"   ✅ SigLIP data saved to {siglip_output_path}")
        
        print(f"\n📊 Final statistics:")
        print(f"   📄 Original CLIP file: {CLIP_KEYFRAMES_FILE} (unchanged)")
        print(f"   📄 Filtered SigLIP file: {siglip_output_path} ({filtered_siglip_data['length']:,} items)")

## Configuration and Run

In [3]:
# Configuration
CLIP_KEYFRAMES_FILE = "/kaggle/input/lucifer-2-clip-batch-1-and-2/all_keyframes_combined.pkl"

LIST_SIGLIP_EMBEDDING = [
    "/kaggle/input/embedding-siglip-kf1-new/SigLIP-kf1-new/embedding_info.pkl",
    "/kaggle/input/embedding-siglip-kf2-new/SigLIP-kf2-new/embedding_info.pkl",
    "/kaggle/input/embedding-siglip-kf3-new/SigLIP-kf3-new/embedding_info.pkl",
    "/kaggle/input/embedding-siglip-kf4-new/SigLIP-kf4-new/embedding_info.pkl",
    "/kaggle/input/embedding-siglip-kf5-new/SigLIP-kf5-new/embedding_info.pkl",
    "/kaggle/input/emb-siglip-kf6-new/SigLIP-kf6-new/embedding_info.pkl",
    "/kaggle/input/emb-siglip-kf7-new/SigLIP-kf7-new/embedding_info.pkl",
    "/kaggle/input/emb-siglip-kf8-new/SigLIP-kf8-new/embedding_info.pkl",
    "/kaggle/input/emb-siglip-kf9-new/SigLIP-kf9-new/embedding_info.pkl",
    "/kaggle/input/emb-siglip-kf10-new/SigLIP-kf10-new/embedding_info.pkl",
    "/kaggle/input/emb-siglip-kf11-new/SigLIP-kf11-new/embedding_info.pkl",
    "/kaggle/input/emb-siglip-kf12-new/SigLIP-kf12-new/embedding_info.pkl",
    "/kaggle/input/emb-siglip-kf13-new/SigLIP-kf13-new/embedding_info.pkl",
    "/kaggle/input/emb-siglip-kf14-new/SigLIP-kf14-new/embedding_info.pkl",
    "/kaggle/input/emb-siglip-kf15-new/SigLIP-kf15-new/embedding_info.pkl"
    # Add more SigLIP embedding files as needed
]

FILTERED_SIGLIP_OUTPUT = "filtered_siglip_embeddings.pkl"

# Initialize filter
filter_tool = SigLIPEmbeddingFilter()

try:
    # Step 1: Load CLIP keyframes
    filter_tool.load_clip_keyframes(CLIP_KEYFRAMES_FILE)
    
    # Step 2: Load and combine SigLIP embeddings
    filter_tool.load_siglip_embeddings(LIST_SIGLIP_EMBEDDING)
    
    # Step 3: Filter SigLIP embeddings based on CLIP paths
    filtered_siglip_data = filter_tool.filter_siglip_by_clip_paths()
    
    # Step 4: Save only filtered SigLIP data (CLIP data is already available)
    filter_tool.save_filtered_siglip(filtered_siglip_data, FILTERED_SIGLIP_OUTPUT)
    
    print(f"\n🎉 Filtering completed successfully!")
    print(f"📄 Use original CLIP file: {CLIP_KEYFRAMES_FILE}")
    print(f"📄 Use filtered SigLIP file: {FILTERED_SIGLIP_OUTPUT}")
    
except Exception as e:
    print(f"❌ Error during filtering: {e}")

📂 Loading CLIP keyframes from /kaggle/input/lucifer-2-clip-batch-1-and-2/all_keyframes_combined.pkl...
✅ CLIP data loaded:
   📊 Paths: 599,392
   📊 Embeddings: 599,392
   📊 Length: 599,392
📂 Loading 15 SigLIP embedding files...
   📄 Loading file 1/15: /kaggle/input/embedding-siglip-kf1-new/SigLIP-kf1-new/embedding_info.pkl


/tmp/ipykernel_10/4268502612.py:51: DeprecationWarning: numpy.core.numeric is deprecated and has been renamed to numpy._core.numeric. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the case, use the public NumPy API. If not, you are using NumPy internals. If you would still like to access an internal attribute, use numpy._core.numeric._frombuffer.
  siglip_data = pickle.load(f)


      ✅ Added 139,457 embeddings
   📄 Loading file 2/15: /kaggle/input/embedding-siglip-kf2-new/SigLIP-kf2-new/embedding_info.pkl
      ✅ Added 78,421 embeddings
   📄 Loading file 3/15: /kaggle/input/embedding-siglip-kf3-new/SigLIP-kf3-new/embedding_info.pkl
      ✅ Added 99,432 embeddings
   📄 Loading file 4/15: /kaggle/input/embedding-siglip-kf4-new/SigLIP-kf4-new/embedding_info.pkl
      ✅ Added 105,695 embeddings
   📄 Loading file 5/15: /kaggle/input/embedding-siglip-kf5-new/SigLIP-kf5-new/embedding_info.pkl
      ✅ Added 55,927 embeddings
   📄 Loading file 6/15: /kaggle/input/emb-siglip-kf6-new/SigLIP-kf6-new/embedding_info.pkl
      ✅ Added 98,823 embeddings
   📄 Loading file 7/15: /kaggle/input/emb-siglip-kf7-new/SigLIP-kf7-new/embedding_info.pkl
      ✅ Added 98,739 embeddings
   📄 Loading file 8/15: /kaggle/input/emb-siglip-kf8-new/SigLIP-kf8-new/embedding_info.pkl
      ✅ Added 98,350 embeddings
   📄 Loading file 9/15: /kaggle/input/emb-siglip-kf9-new/SigLIP-kf9-new/embedding

Filtering: 100%|██████████| 599392/599392 [00:00<00:00, 938113.40it/s] 



📊 Filtering results:
   ✅ Matched paths: 599,392
   ❌ Missing paths: 0
   📈 Match rate: 100.0%
💾 Saving filtered SigLIP data...
   ✅ SigLIP data saved to filtered_siglip_embeddings.pkl

📊 Final statistics:
   📄 Original CLIP file: /kaggle/input/lucifer-2-clip-batch-1-and-2/all_keyframes_combined.pkl (unchanged)
   📄 Filtered SigLIP file: filtered_siglip_embeddings.pkl (599,392 items)

🎉 Filtering completed successfully!
📄 Use original CLIP file: /kaggle/input/lucifer-2-clip-batch-1-and-2/all_keyframes_combined.pkl
📄 Use filtered SigLIP file: filtered_siglip_embeddings.pkl


## Validation

Verify that the filtered CLIP and SigLIP data have matching paths and lengths.

In [4]:
# Validation: Load and compare filtered data
def validate_filtered_data(clip_file, siglip_file):
    """
    Validate that original CLIP and filtered SigLIP data have matching paths and lengths.
    """
    print(f"🔍 Validating filtered data...")
    
    # Load data
    with open(clip_file, 'rb') as f:
        clip_data = pickle.load(f)
    
    with open(siglip_file, 'rb') as f:
        siglip_data = pickle.load(f)
    
    print(f"\n📊 Data loaded:")
    print(f"   📄 CLIP (original): {len(clip_data['paths']):,} paths, {len(clip_data['embeddings']):,} embeddings, length: {clip_data['length']:,}")
    print(f"   📄 SigLIP (filtered): {len(siglip_data['paths']):,} paths, {len(siglip_data['embeddings']):,} embeddings, length: {siglip_data['length']:,}")
    
    # Validation checks
    checks_passed = 0
    total_checks = 4
    
    # Check 1: Same number of paths
    if len(clip_data['paths']) == len(siglip_data['paths']):
        print(f"✅ Check 1: Same number of paths ({len(clip_data['paths']):,})")
        checks_passed += 1
    else:
        print(f"❌ Check 1: Different number of paths (CLIP: {len(clip_data['paths']):,}, SigLIP: {len(siglip_data['paths']):,})")
    
    # Check 2: Same length values
    if clip_data['length'] == siglip_data['length']:
        print(f"✅ Check 2: Same length values ({clip_data['length']:,})")
        checks_passed += 1
    else:
        print(f"❌ Check 2: Different length values (CLIP: {clip_data['length']:,}, SigLIP: {siglip_data['length']:,})")
    
    # Check 3: Paths are identical
    if clip_data['paths'] == siglip_data['paths']:
        print(f"✅ Check 3: Paths are identical")
        checks_passed += 1
    else:
        print(f"❌ Check 3: Paths are not identical")
        # Show first few differences
        differences = []
        for i, (clip_path, siglip_path) in enumerate(zip(clip_data['paths'], siglip_data['paths'])):
            if clip_path != siglip_path:
                differences.append((i, clip_path, siglip_path))
                if len(differences) >= 3:
                    break
        
        if differences:
            print(f"      First few differences:")
            for i, clip_path, siglip_path in differences:
                print(f"        Index {i}: CLIP='{clip_path}' vs SigLIP='{siglip_path}'")
    
    # Check 4: Data consistency within each dataset
    clip_consistent = len(clip_data['paths']) == len(clip_data['embeddings']) == clip_data['length']
    siglip_consistent = len(siglip_data['paths']) == len(siglip_data['embeddings']) == siglip_data['length']
    
    if clip_consistent and siglip_consistent:
        print(f"✅ Check 4: Internal data consistency")
        checks_passed += 1
    else:
        print(f"❌ Check 4: Internal data inconsistency")
        if not clip_consistent:
            print(f"      CLIP: {len(clip_data['paths'])} paths, {len(clip_data['embeddings'])} embeddings, {clip_data['length']} length")
        if not siglip_consistent:
            print(f"      SigLIP: {len(siglip_data['paths'])} paths, {len(siglip_data['embeddings'])} embeddings, {siglip_data['length']} length")
    
    # Summary
    print(f"\n📈 Validation Summary: {checks_passed}/{total_checks} checks passed")
    
    if checks_passed == total_checks:
        print(f"🎉 All validation checks passed! Data is ready for use.")
        print(f"📄 CLIP embeddings: {clip_file}")
        print(f"📄 SigLIP embeddings: {siglip_file}")
    else:
        print(f"⚠️  Some validation checks failed. Please review the data.")
    
    return checks_passed == total_checks

# Run validation
try:
    is_valid = validate_filtered_data(CLIP_KEYFRAMES_FILE, FILTERED_SIGLIP_OUTPUT)
except Exception as e:
    print(f"❌ Error during validation: {e}")

🔍 Validating filtered data...

📊 Data loaded:
   📄 CLIP (original): 599,392 paths, 599,392 embeddings, length: 599,392
   📄 SigLIP (filtered): 599,392 paths, 599,392 embeddings, length: 599,392
✅ Check 1: Same number of paths (599,392)
✅ Check 2: Same length values (599,392)
✅ Check 3: Paths are identical
✅ Check 4: Internal data consistency

📈 Validation Summary: 4/4 checks passed
🎉 All validation checks passed! Data is ready for use.
📄 CLIP embeddings: /kaggle/input/lucifer-2-clip-batch-1-and-2/all_keyframes_combined.pkl
📄 SigLIP embeddings: filtered_siglip_embeddings.pkl
